##### Import the required modules and configure the system path to locate them

In [1]:
import sys
sys.path.append("../../utils")
import networkx
import yaml
import os
import pandas as pd
import astra_sim_sdk.astra_sim_sdk as astra_sim_kit
from common import FileFolderUtils
from astra_sim import AstraSim, Collective, NetworkBackend
from astra_sim_sdk import Device, Component
from infragraph import Component, InfrastructureEdge
from infragraph.infragraph_service import InfraGraphService
from infragraph.blueprints.devices.generic.server import Server
from infragraph.blueprints.devices.generic.generic_switch import Switch

##### Call the AstraSim client helper with the server endpoint and tag to connect to the ASTRA-sim gRPC server, initialize the SDK, and create a tagged folder for configs, results, and logs

In [2]:
astra = AstraSim(server_endpoint="172.17.0.2:8989", tag = "ns3_single_tier_with_generic_server")

Resetting test directory
All contents of the folder /workspaces/astra_sim_service/client-scripts/utils/../trial/ns3_single_tier_with_generic_server have been deleted.
Successfully connected to gRPC server at 172.17.0.2:8989


##### Generate workload execution traces for each rank and set the required data size for AstraSim configuration

In [3]:
astra.configuration.common_config.workload = astra.generate_collective(collective=Collective.ALLREDUCE, coll_size= 8 *1024*1024, npu_range=[0,8])
print(astra.configuration.common_config.workload)

Generated 8 et in /workspaces/astra_sim_service/client-scripts/utils/../trial/ns3_single_tier_with_generic_server/configuration/workload
/workspaces/astra_sim_service/client-scripts/utils/../trial/ns3_single_tier_with_generic_server/configuration/workload/ns3_single_tier_with_generic_server


##### Configure the ASTRA-sim system config

In [4]:
astra.configuration.common_config.system.scheduling_policy = astra.configuration.common_config.system.LIFO
astra.configuration.common_config.system.endpoint_delay = 10
astra.configuration.common_config.system.active_chunks_per_dimension = 1
astra.configuration.common_config.system.all_gather_implementation = [astra.configuration.common_config.system.RING]
astra.configuration.common_config.system.all_to_all_implementation = [astra.configuration.common_config.system.DIRECT]
astra.configuration.common_config.system.all_reduce_implementation = [astra.configuration.common_config.system.ONERING]
astra.configuration.common_config.system.collective_optimization = astra.configuration.common_config.system.LOCALBWAWARE
astra.configuration.common_config.system.local_mem_bw = 1600
print(astra.configuration.common_config.system)

active_chunks_per_dimension: 1
all_gather_implementation:
- ring
all_reduce_implementation:
- oneRing
all_to_all_implementation:
- direct
collective_optimization: localBWAware
endpoint_delay: 10
local_mem_bw: 1600
local_reduction_delay: 0
preferred_dataset_splits: 1
reduce_scatter_implementation:
- ring
scheduling_policy: LIFO
trace_enabled: 0



##### Configure ASTRA-sim remote memory configuration

In [5]:
astra.configuration.common_config.remote_memory.memory_type = astra.configuration.common_config.remote_memory.NO_MEMORY_EXPANSION
print(astra.configuration.common_config.remote_memory)

memory_type: NO_MEMORY_EXPANSION
remote_mem_bw: 0
remote_mem_latency: 0



##### Configure the selected network backend and the topology (infragraph or nc_topology)

In [6]:
# We need to configure the network backend here since we are translating the topology from infragraph and not creating it directly from the sdk.
astra.configuration.network_backend.choice = astra.configuration.network_backend.NS3
astra.configuration.network_backend.ns3.topology.choice = astra.configuration.network_backend.ns3.topology.INFRAGRAPH
astra.configuration.network_backend.ns3.network.packet_payload_size = int(8192)
astra.configuration.network_backend.ns3.logical_topology.logical_dimensions = [8]
astra.configuration.network_backend.ns3.trace.trace_ids = [0, 1, 2, 3, 4, 5, 6, 7]

##### Creating Infrastructure with four host and one rack Device

In [7]:
astra.configuration.infragraph.infrastructure.name = "1host-4ranks"

server = Device()
server.deserialize((Server(npu_factor=1).serialize()))

hosts = astra.configuration.infragraph.infrastructure.instances.add(
    name="host", device=server.name, count=4
)
switch = Device()
switch.deserialize(Switch(port_count=16).serialize())

rack_switch = astra.configuration.infragraph.infrastructure.instances.add(
    name="rack_switch", device=switch.name, count=1
)

astra.configuration.infragraph.infrastructure.devices.append(server).append(switch)



##### Creating Links

In [8]:
rack_link = astra.configuration.infragraph.infrastructure.links.add(
    name="rack-link",
    description="Link characteristics for connectivity between servers and rack switch",
)
rack_link.physical.bandwidth.gigabits_per_second = 200

##### Adding edges and annotations

In [9]:
host_component = InfraGraphService.get_component(server, Component.NIC)
switch_component = InfraGraphService.get_component(switch, Component.PORT)
# link each host to one leaf switch
for idx in range(hosts.count):
    edge = astra.configuration.infragraph.infrastructure.edges.add(
        scheme=InfrastructureEdge.ONE2ONE, link=rack_link.name
    )
    edge.ep1.instance = f"{hosts.name}[{idx}]"
    edge.ep1.component = f"{host_component.name}[0]"
    edge.ep2.instance = f"{rack_switch.name}[0]"
    edge.ep2.component = f"{switch_component.name}[{idx * 2}]"
    edge = astra.configuration.infragraph.infrastructure.edges.add(
        scheme=InfrastructureEdge.ONE2ONE, link=rack_link.name
    )
    edge.ep1.instance = f"{hosts.name}[{idx}]"
    edge.ep1.component = f"{host_component.name}[1]"
    edge.ep2.instance = f"{rack_switch.name}[0]"
    edge.ep2.component = f"{switch_component.name}[{idx * 2 + 1}]"

# annotation
host_device_spec = astra_sim_kit.AnnotationDeviceSpecifications()
host_device_spec.device_bandwidth_gbps = 200
host_device_spec.device_latency_ms = 0.05
host_device_spec.device_name = "server"
host_device_spec.device_type = "host"
astra.configuration.infragraph.annotations.device_specifications.append(host_device_spec)

switch_device_spec = astra_sim_kit.AnnotationDeviceSpecifications()
switch_device_spec.device_bandwidth_gbps = 200
switch_device_spec.device_latency_ms = 0.05
switch_device_spec.device_name = "switch"
switch_device_spec.device_type = "switch"
astra.configuration.infragraph.annotations.device_specifications.append(
    switch_device_spec
)

##### Configure ASTRA-sim cmd parameters

In [10]:
astra.configuration.common_config.cmd_parameters.comm_scale = 1
astra.configuration.common_config.cmd_parameters.injection_scale = 1
astra.configuration.common_config.cmd_parameters.rendezvous_protocol = False

#### Start the simulation by specifying the network backend

In [11]:
astra.run_simulation(NetworkBackend.NS3)

Generating Configuration ZIP now
output_path: /workspaces/astra_sim_service/client-scripts/utils/../trial/ns3_single_tier_with_generic_server/config.zip
folder_path: /workspaces/astra_sim_service/client-scripts/utils/../trial/ns3_single_tier_with_generic_server/configuration/workload/..
pack_zip complete
message: 'Configuration applied successfully. warnings: Unable to generate communicator
  group message from schema - communicator group configuration empty'

message: Simulation started successfully

astra-sim server Status: running
Transferring Files from ASTRA-sim server
All files downloaded Successfully
Translating Metrics...
Generated fct.csv at:  /workspaces/astra_sim_service/client-scripts/utils/../trial/ns3_single_tier_with_generic_server/output/fct.csv
Generated: flow_stats.csv at:  /workspaces/astra_sim_service/client-scripts/utils/../trial/ns3_single_tier_with_generic_server/output/flow_stats.csv
All metrics translated successfully
Simulation completed


/workspaces/astra_sim_service/client-scripts/notebooks/infragraph/../../utils/common.py:274: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(
/workspaces/astra_sim_service/client-scripts/notebooks/infragraph/../../utils/common.py:237: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(


##### Download all the configurations as a zip

In [12]:
astra.download_configuration()

Downloaded all configuration in /workspaces/astra_sim_service/client-scripts/utils/../trial/ns3_single_tier_with_generic_server/server_configuration.zip


##### Read output files

In [13]:
df = pd.read_csv(os.path.join(FileFolderUtils.get_instance().OUTPUT_DIR, "fct.csv"))
df.head()

df = pd.read_csv(os.path.join(FileFolderUtils.get_instance().OUTPUT_DIR, "flow_stats.csv"))
df.head()

,Source ip,Destination ip,Source Port,Destination Port,Data size (B),Start Time,FCT,Standalone FCT,Total Bytes Tx,Total Bytes Rx,Completion time (ms),Start (ms),End (ms)
0,11.0.7.1,11.0.0.1,10000,100,1048576,10,42953,42842,1048576,1048576,0.042953,0.00001,0.042963
1,11.0.3.1,11.0.4.1,10000,100,1048576,10,42953,42842,1048576,1048576,0.042953,0.00001,0.042963
2,11.0.0.1,11.0.1.1,10000,100,1048576,10,42955,42842,1048576,1048576,0.042955,0.00001,0.042965
3,11.0.4.1,11.0.5.1,10000,100,1048576,10,42955,42842,1048576,1048576,0.042955,0.00001,0.042965
4,11.0.1.1,11.0.2.1,10000,100,1048576,10,43280,42842,1048576,1048576,0.043280,0.00001,0.043290


##### Save infragraph as a yaml

In [14]:
with open(os.path.join(FileFolderUtils.get_instance().OUTPUT_DIR,"../infrastructure","ns3_single_tier_with_dgx.yaml"),"w") as f:
    data = astra.configuration.infragraph.infrastructure.serialize("dict")
    yaml.dump(data, f, default_flow_style=False, indent=4)

print("saved yaml to:", os.path.join(FileFolderUtils.get_instance().OUTPUT_DIR,"..","ns3_single_tier_with_dgx.yaml"))

saved yaml to: /workspaces/astra_sim_service/client-scripts/utils/../trial/ns3_single_tier_with_generic_server/output/../ns3_single_tier_with_dgx.yaml
